1 - explore

In [0]:
import requests, json

TRAFA = "https://api.trafa.se/api/data"

def trafa(query):
    r = requests.get(TRAFA, params={"query": query, "lang": "sv"}, timeout=60)
    r.raise_for_status()
    return r.json()

# A) one county, all its kommuner (no regkom values listed), 2 years, BEV + total
a = trafa("t10026|ar:2024,2025|reglan:01|regkom|drivmedel:103,t1|itrfslut")
print("Top-level keys:", list(a.keys()))
rows = a.get("Rows", [])
print("A) rows:", len(rows), "  (Stockholm county has 26 kommuner -> expect 26 × 2 × 2 = 104)")
print(json.dumps(rows[:2], ensure_ascii=False, indent=2)[:1500])

# B) same, but all fuel types (no drivmedel values listed)
b = trafa("t10026|ar:2025|reglan:01|regkom|drivmedel|itrfslut")
print("B) rows:", len(b.get("Rows", [])), "  (expect 26 × 9 = 234 if 'Totalt' is included)")

2 - explore regions

In [0]:
from collections import Counter

def trafa_rows(resp):
    """Rows -> dicts: dimensions keep code (Name) + label (Value); measures keep the raw Value."""
    out = []
    for row in resp.get("Rows", []):
        rec = {}
        for c in row["Cell"]:
            if c["IsMeasure"]:
                rec[c["Column"]] = c["Value"]
            else:
                rec[c["Column"]] = c["Name"]
                rec[c["Column"] + "_label"] = c["Value"]
        out.append(rec)
    return out

ra, rb = trafa_rows(a), trafa_rows(b)

print("A regkom (code, label): count")
for k, n in sorted(Counter((r["regkom"], r["regkom_label"]) for r in ra).items()):
    if n != 4: print("   ", k, n)          # only the odd ones; normal kommuner have 4
print("A distinct regkom:", len({r["regkom"] for r in ra}))

print("\nB drivmedel:", sorted(Counter((r["drivmedel"], r["drivmedel_label"]) for r in rb).items()))
print("B rows per regkom (odd ones):",
      {k: n for k, n in Counter((r["regkom"], r["regkom_label"]) for r in rb).items() if n != 9})
print("B empty measure values:", sum(1 for r in rb if r.get("itrfslut") in (None, "")))
print("Errors:", a.get("Errors"), "| Notes:", a.get("Notes"))

3 - find codes for kommun

In [0]:
for code, label in sorted({(r["regkom"], r["regkom_label"]) for r in ra}):
    print(code, label)

4 loader

In [0]:
import time
from pathlib import Path

ROOT_T = Path("/Volumes/laddstolpar_df/landing/raw/trafa/T10026")
YEARS  = [str(y) for y in range(2020, 2026)]                    # 2020–2025
LAN    = sorted(r[0] for r in spark.sql("""
            SELECT DISTINCT region FROM laddstolpar_df.bronze.scb_tab628
            WHERE length(region) = 2 AND region <> '00'""").collect())
assert len(LAN) == 21, LAN                                       # TAB628 has no 15/16

def land_trafa(years=YEARS, lan=LAN, pause_s=0.5):
    folder = ROOT_T / f"ar={years[0]}-{years[-1]}"
    stats = {"calls": 0, "written": 0, "skipped": 0, "rows": 0}
    for l in lan:
        p = folder / f"reglan={l}.json"
        if p.exists() and p.stat().st_size > 0:                  # idempotency
            stats["skipped"] += 1
            continue
        q = f"t10026|ar:{','.join(years)}|reglan:{l}|regkom|drivmedel|itrfslut"
        r = requests.get(TRAFA, params={"query": q, "lang": "sv"}, timeout=120)
        r.raise_for_status()                                     # on error: rerun, finished counties are skipped
        js = r.json()
        assert not js.get("Errors"), f"{l}: {js.get('Errors')}"
        n = len(js.get("Rows", []))
        assert n > 0, f"{l}: 0 rows"
        folder.mkdir(parents=True, exist_ok=True)
        p.write_text(r.text, encoding="utf-8")                   # raw response, unchanged
        stats["calls"] += 1; stats["written"] += 1; stats["rows"] += n
        time.sleep(pause_s)
    return stats

print(land_trafa())

5 - load into bronze

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

T_COLS = ["ar", "ar_label", "reglan", "reglan_label", "regkom", "regkom_label",
          "drivmedel", "drivmedel_label", "itrfslut"]
T_SCHEMA = StructType([StructField(c, StringType()) for c in T_COLS])

def bronze_trafa():
    target = "laddstolpar_df.bronze.trafa_t10026"
    done = set()
    if spark.catalog.tableExists(target):                         # processed-files ledger
        done = {r[0] for r in spark.table(target).select("_file").distinct().collect()}
    files = sorted(str(p) for p in ROOT_T.rglob("*.json"))
    stats = {"table": target, "files_total": len(files), "new_files": 0, "rows_written": 0}
    for f in files:
        if f in done:
            continue
        rows = trafa_rows(json.loads(Path(f).read_text(encoding="utf-8")))
        got = set(rows[0])
        assert got == set(T_COLS), f"{f}: columns differ – missing {set(T_COLS)-got}, extra {got-set(T_COLS)}"
        df = (spark.createDataFrame([tuple(r.get(c) for c in T_COLS) for r in rows], T_SCHEMA)
                .withColumn("_snapshot", F.lit(re.search(r"(ar=[0-9-]+)", f).group(1)))
                .withColumn("_file", F.lit(f))
                .withColumn("_ingested_at", F.current_timestamp()))
        df.write.mode("append").saveAsTable(target)               # one commit per file
        stats["new_files"] += 1
        stats["rows_written"] += len(rows)
    return stats

print(bronze_trafa())

6 - Check tables

In [0]:
%sql
SELECT COUNT(*)                                            AS rows_,
       COUNT(DISTINCT _file)                               AS files,
       COUNT(DISTINCT CASE WHEN regkom <> 't1' THEN regkom END) AS kommuner,
       COUNT_IF(regkom = 't1')                             AS county_total_rows,
       COUNT_IF(itrfslut IS NULL OR itrfslut = '')         AS empty_values,
       COUNT_IF(NOT itrfslut RLIKE '^[0-9]+$')             AS non_numeric
FROM laddstolpar_df.bronze.trafa_t10026;

-- Same kommuner in both sources? (expect 0 rows)
WITH t AS (SELECT DISTINCT regkom AS code FROM laddstolpar_df.bronze.trafa_t10026 WHERE regkom <> 't1'),
     s AS (SELECT DISTINCT region AS code FROM laddstolpar_df.bronze.scb_tab628 WHERE length(region) = 4)
SELECT 'only in Trafikanalys' AS side, code FROM t WHERE code NOT IN (SELECT code FROM s)
UNION ALL
SELECT 'only in SCB', code FROM s WHERE code NOT IN (SELECT code FROM t);